# Промпт для оценки релевантности музыкальной выдачи (GigaChat-Pro)

Тестовое задание на промпт-инженера. Модель получает запрос пользователя, название трека и исполнителя из поисковой выдачи и ставит одну из четырёх меток: `vital`, `rel_plus`, `rel_minus`, `notrel`.

Валидационный набор `valid_set.tsv` (105 примеров) в репозиторий не входит. Ключ GigaChat берётся из переменной окружения `GIGACHAT_CREDENTIALS`.

In [1]:
# валидационный сет для разработки и дальнейшей корректировки промпта
import pandas as pd

test = pd.read_csv('valid_set.tsv', sep='\t')
print(f'Размер сета: {len(test)} строк')
print('\nРаспределение классов:')
print(test['golden_answer'].value_counts())


In [ ]:
test.sample()

In [ ]:
# требуется установить пакет для дальнейшего использования модели GigaChat
! pip install -U langchain-gigachat

In [ ]:
! pip install langchain-core

In [5]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_gigachat import GigaChat

# Авторизация в GigaChat
# temperature=0 — задача классификационная, нужны воспроизводимые ответы.
# max_tokens=10 — метка занимает 1–3 токена, 10 с запасом. Страхует от того,
# что модель уйдёт в простыню, и экономит время.
llm = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"], #Ключ авторизации
    scope="GIGACHAT_API_PERS",
    model="GigaChat-Pro",
    # Отключает проверку наличия сертификатов НУЦ Минцифры
    verify_ssl_certs=False,
    streaming=False,
    temperature=0,
    max_tokens=10,
)


In [28]:
system_prompt = """
Ты — асессор музыкального поискового сервиса. На вход тебе подаются:
- запрос пользователя,
- название найденного трека,
- исполнитель найденного трека.

Твоя задача — оценить, насколько результат поиска соответствует запросу, и выбрать РОВНО ОДНУ метку из четырёх:

1) vital — пользователь, скорее всего, нашёл именно то, что искал.
   Используй эту метку, когда:
   • из запроса понятно, что нужна именно эта песня именно этого исполнителя, и в выдаче ровно она;
   • в запросе указан и трек, и исполнитель — оба совпадают с выдачей;
   • запрошен саундтрек из конкретного фильма/мультфильма/сериала/игры, и название фильма присутствует в названии найденного трека;
   • найденная версия трека идентична запрошенной;
   • запрос по тексту песни с указанием исполнителя, и у этого исполнителя ровно одна песня с таким текстом.

2) rel_plus — выдача полезная, пользователь, скорее всего, доволен.
   Используй эту метку, когда:
   • запрошено название песни без указания исполнителя, найдена песня с этим названием (даже если популярных версий несколько);
   • запрос только по исполнителю — найдена песня этого исполнителя;
   • запрос по жанру/настроению/тематике — найдена песня в этом жанре;
   • запрос по альбому — найдена песня из этого альбома;
   • запрос по тексту/смыслу — найдена песня с этим текстом или близким смыслом;
   • запрос по саундтреку из фильма без конкретного названия — найден один из саундтреков этого фильма;
   • запрошено несколько исполнителей вместе — найден их совместный трек;
   • запрошен один исполнитель, найден его совместный трек с другими (есть и сольная версия в каталоге, и коллаб);
   • из запроса не до конца ясно, что хотел пользователь, но трек в принципе подходит;
   • пользователь, видимо, ошибся в авторе, но очевидно хотел именно этот трек.

3) rel_minus — выдача могла быть лучше, но не совсем мимо.
   Используй эту метку, когда:
   • существует более популярная и широко известная песня/исполнитель с похожим названием, и пользователь явно хотел её (опечатка/транслит/похожее звучание);
   • запрошено несколько исполнителей вместе — найден трек только одного из них;
   • запрошен один исполнитель соло — найден только его коллаб с другими (сольной версии нет);
   • запрошена конкретная версия трека — найдена другая, но близкая версия.

4) notrel — выдача совсем не та, ответ бесполезен.
   Используй эту метку, когда:
   • найден совершенно другой трек/исполнитель, не упомянутый и не подразумеваемый в запросе;
   • в запросе явно указано название песни/исполнитель, в выдаче ни одно из них не совпадает по смыслу;
   • запрошен саундтрек из конкретного фильма — найдена не из того фильма;
   • жанр/тематика выдачи не соответствует запросу.

Правила вывода:
- Внимательно сравнивай запрос с парой (трек, исполнитель).
- Учитывай возможные опечатки и транслит в запросе пользователя.
- Если в запросе указан и трек, и исполнитель, и оба совпадают — это vital.
- Если совпало только название песни без исполнителя — обычно rel_plus.
- Если есть более известная одноимённая песня или исполнитель — rel_minus.
- Если ни трек, ни исполнитель не отвечают запросу по смыслу — notrel.

Примеры:

Запрос пользователя: цой звезда по имени солнце
Найденный трек: Звезда по имени Солнце
Исполнитель: Виктор Цой
Оценка: vital

Запрос пользователя: поставь песню облака
Найденный трек: Облака
Исполнитель: Звери
Оценка: rel_plus

Запрос пользователя: включи группу металлика
Найденный трек: Enter Sandman
Исполнитель: Metallica
Оценка: rel_plus

Запрос пользователя: включи розенбаума
Найденный трек: Розенбаум
Исполнитель: Михаил Шуфутинский
Оценка: rel_minus

Запрос пользователя: включи земфиру хочешь
Найденный трек: ХОЧЕШЬ?
Исполнитель: Лёша Свик
Оценка: rel_minus

Запрос пользователя: включи rammstein du hast
Найденный трек: Москва
Исполнитель: Yanix
Оценка: notrel

В ответе верни ТОЛЬКО одну из четырёх меток: vital, rel_plus, rel_minus, notrel.
Без объяснений, без кавычек, без знаков препинания, без дополнительного текста.
""".strip()


In [29]:
# user_prompt формируется для каждой строки из (query, track_name, author)
def build_user_prompt(query: str, track_name: str, author: str) -> str:
    return (
        f"Запрос пользователя: {query}\n"
        f"Найденный трек: {track_name}\n"
        f"Исполнитель: {author}\n"
        f"Оценка:"
    )

# Пример
print(build_user_prompt(test.iloc[0]['query'],
                        test.iloc[0]['track_name'],
                        test.iloc[0]['author']))


In [30]:
# Генерация ответов модели по всему валидационному сету
from tqdm import tqdm

raw_predictions = []  # сюда складываем сырые текстовые ответы модели

for _, row in tqdm(test.iterrows(), total=len(test)):
    user_prompt = build_user_prompt(row['query'], row['track_name'], row['author'])
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    try:
        res = llm.invoke(messages)
        raw_predictions.append(res.content)
    except Exception as e:
        # На случай сетевых/квотных ошибок — не падаем, помечаем строку
        raw_predictions.append(f"__ERROR__: {e}")

test['raw_prediction'] = raw_predictions
test.head()


KeyboardInterrupt: 

In [ ]:
# Превью: запрос, выдача, эталон и сырой ответ модели
test[['query', 'track_name', 'author', 'golden_answer', 'raw_prediction']].head(10)


In [ ]:
# Парсинг сырого ответа модели в одну из четырёх меток.
# Берём первую встретившуюся валидную метку в нижнем регистре.
import re

VALID_LABELS = ['vital', 'rel_plus', 'rel_minus', 'notrel']

def parse_label(raw: str) -> str:
    if not isinstance(raw, str):
        return 'unknown'
    text = raw.strip().lower()
    # Сначала ищем точное совпадение по словарю в порядке убывания "длины",
    # чтобы rel_plus / rel_minus не путались с rel.
    for label in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(r'\b' + re.escape(label) + r'\b', text):
            return label
    return 'unknown'

test['prediction'] = test['raw_prediction'].apply(parse_label)
print('Распределение предсказаний:')
print(test['prediction'].value_counts())
print('\nДоля нераспарсенных ответов:',
      (test['prediction'] == 'unknown').mean())


In [ ]:
# Метрики качества промпта
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import pandas as pd

LABELS = ['vital', 'rel_plus', 'rel_minus', 'notrel']

y_true = test['golden_answer']
y_pred = test['prediction']

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=LABELS, average='weighted', zero_division=0)

print(f'Accuracy:    {acc:.3f}')
print(f'Macro F1:    {macro_f1:.3f}')
print(f'Weighted F1: {weighted_f1:.3f}')

print('\nClassification report:')
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0, digits=3))

print('\nConfusion matrix (строки — golden, столбцы — prediction):')
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))


### Почему выбраны такие метрики

Задача — мультиклассовая классификация на 4 классa с заметным дисбалансом
(`rel_plus`: 39, `vital`: 28, `notrel`: 22, `rel_minus`: 16).

- **Accuracy** — общий процент верных оценок. Простая базовая метрика,
  но при дисбалансе классов может быть оптимистичной (модель, угадывающая
  только мажоритарный класс, получит ~37%).
- **Macro F1** — F1, усреднённый по классам без учёта их размера. Главная
  метрика для нас: миноритарный `rel_minus` важен так же, как мажоритарный
  `rel_plus`. Если модель плохо работает на одном из классов, macro F1 это сразу
  покажет.
- **Weighted F1** — F1, взвешенный по размеру класса. Показывает качество
  с точки зрения «средней» строки в выборке.
- **Classification report** — precision/recall/F1 по каждому классу. Помогает
  увидеть, где именно модель ошибается (например, путает `vital` с `rel_plus`).
- **Confusion matrix** — детальная картина ошибок: какие классы между собой
  путаются. На практике частые путаницы будут между соседними оценками
  (`vital` ↔ `rel_plus` и `rel_plus` ↔ `rel_minus`), они менее критичны,
  чем путаница `vital` ↔ `notrel`.

Ориентир для интерпретации: при 4 классах случайное угадывание даёт
accuracy ≈ 0.25 и macro F1 ≈ 0.20–0.25. Целимся хотя бы в macro F1 ≥ 0.55–0.6
для прода — это означает, что промпт сравним с разметкой средненького асессора.


# **<span style="background-color: #fff69d">Self-consistency (опционально)</span>**

Запускаем тот же промпт `n` раз с `temperature=0.3` и берём мажоритарную метку
(majority vote). Идея: на пограничных случаях модель «колеблется» между двумя
соседними классами; усреднение по нескольким прогонам стабилизирует ответ
и обычно даёт +2–4 п.п. к macro F1 на классификационных задачах.

Стоимость: в `n` раз дороже и медленнее бейзлайна. Включать стоит, только если
метрики первой версии не дотягивают до целевых.


In [ ]:
import os
# Отдельный клиент с ненулевой температурой — иначе все n прогонов вернут
# идентичный ответ, и majority vote становится бессмысленным.
llm_sc = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"],  # Ключ авторизации (тот же)
    scope="GIGACHAT_API_PERS",
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    streaming=False,
    temperature=0.3,
    max_tokens=10,
)

N_SAMPLES = 5  # число прогонов на одну строку. Чем больше, тем стабильнее, но дороже.


In [ ]:
# Self-consistency: n прогонов на строку + majority vote
from collections import Counter

sc_raw_predictions = []  # list[list[str]] — n сырых ответов на каждую строку
sc_predictions = []      # list[str] — итоговая метка после голосования

for _, row in tqdm(test.iterrows(), total=len(test)):
    user_prompt = build_user_prompt(row['query'], row['track_name'], row['author'])
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    samples = []
    for _ in range(N_SAMPLES):
        try:
            res = llm_sc.invoke(messages)
            samples.append(res.content)
        except Exception as e:
            samples.append(f"__ERROR__: {e}")
    sc_raw_predictions.append(samples)

    # парсим каждый ответ, оставляем только валидные метки, голосуем
    parsed = [parse_label(s) for s in samples]
    valid = [p for p in parsed if p in VALID_LABELS]
    if valid:
        # самая частая; при ничьей Counter вернёт ту, что встретилась первой
        sc_predictions.append(Counter(valid).most_common(1)[0][0])
    else:
        sc_predictions.append('unknown')

test['raw_prediction_sc'] = sc_raw_predictions
test['prediction_sc'] = sc_predictions

print('Распределение предсказаний (self-consistency):')
print(test['prediction_sc'].value_counts())
print('\nДоля нераспарсенных:', (test['prediction_sc'] == 'unknown').mean())


In [ ]:
# Сравнение: бейзлайн vs self-consistency
y_true = test['golden_answer']

print('=== Бейзлайн (T=0, single pass) ===')
y_pred_base = test['prediction']
print(f'Accuracy:    {accuracy_score(y_true, y_pred_base):.3f}')
print(f'Macro F1:    {f1_score(y_true, y_pred_base, labels=LABELS, average="macro", zero_division=0):.3f}')
print(f'Weighted F1: {f1_score(y_true, y_pred_base, labels=LABELS, average="weighted", zero_division=0):.3f}')

print('\n=== Self-consistency (T=0.3, n=5, majority vote) ===')
y_pred_sc = test['prediction_sc']
print(f'Accuracy:    {accuracy_score(y_true, y_pred_sc):.3f}')
print(f'Macro F1:    {f1_score(y_true, y_pred_sc, labels=LABELS, average="macro", zero_division=0):.3f}')
print(f'Weighted F1: {f1_score(y_true, y_pred_sc, labels=LABELS, average="weighted", zero_division=0):.3f}')

print('\nClassification report (self-consistency):')
print(classification_report(y_true, y_pred_sc, labels=LABELS, zero_division=0, digits=3))

print('\nConfusion matrix (self-consistency):')
cm_sc = confusion_matrix(y_true, y_pred_sc, labels=LABELS)
print(pd.DataFrame(cm_sc, index=LABELS, columns=LABELS))

# Где self-consistency меняет ответ относительно бейзлайна
changed = test[test['prediction'] != test['prediction_sc']]
print(f'\nИзменено ответов после self-consistency: {len(changed)} из {len(test)}')
print('Из них верных стало:',
      (changed['prediction_sc'] == changed['golden_answer']).sum(),
      '| было верных:',
      (changed['prediction'] == changed['golden_answer']).sum())
